In [33]:
import torch
from split_and_train import *

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ConvNet().to(device)
model.load_state_dict(torch.load("model.pt", map_location=device))

<All keys matched successfully>

In [34]:
import pandas as pd

def get_ecg_10sarray(file_name):
    data = pd.read_parquet(file_name)
    window_size = 10 * 250
    chunks = []
    for i in range(0, len(data), window_size):
        chunk = data.loc[i:i+ window_size-1, 'ecg'].values
        
        if len(chunk) == 2500:
            chunks.append(chunk)
    return chunks

filename = 'ecg_log_2026-03-03_235609.parquet'
chunks = get_ecg_10sarray(filename)
predict_df = pd.DataFrame({"ecg_raw": chunks, "label_int": [0]*len(chunks)})

In [35]:
predict_ds = ECGDataset(predict_df, transform=ecg_transform)
predict_loader = DataLoader(predict_ds, batch_size=64, shuffle=False)

model.eval()
all_probs = []

with torch.no_grad():
    for batch in predict_loader:
        x = batch["data"].to(device)
        logits = model(x)
        probs = torch.sigmoid(logits) # 获取概率
        all_probs.extend(probs.cpu().numpy().flatten())

In [39]:
pred_labels = (np.array(all_probs) >= 0.5).astype(int)

In [44]:
pred_labels[pred_labels == 1]

array([1, 1, 1, ..., 1, 1, 1], shape=(4030,))